In [ ]:
include( "../args.jl" )
include( "../model.jl" )
include( "../cluster.jl" )
include( "../prob.jl" )
include( "../visual.jl" )

include( "../0-params/rho-list.jl" )
include( "../0-params/spd-list.jl" )
include( "../0-params/nos-list.jl" )

using Optim
using ThreadsX

home, base = expfolder( "system" )

colormap = cgrad( :ice; rev=true )

In [ ]:
# Hyper parameters.
N = 1024

# Create folder and extract critical noise.
folderdata = [createfolder( findfolder( N, ρ, s; base=base, warn=false, crit=true ) ) for ρ ∈ ρlist, s ∈ slist]

# Extract critical noise values.
ηcdata = first.( readdlm.( folderdata.*"../order-0.5.txt" ) );

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ*Ns, "Importing data."; dt=1.0 )

# Initialize data variables.
Fdatalist = Matrix{Union{Nothing,Matrix{defInt}}}( undef, Nρ, Ns )

# Import cluster dynamical data.
for i ∈ 1:Nρ, j ∈ 1:Ns
    check = ispath( folderdata[i,j]*"frequency-map.txt" )
    Fdatalist[i,j] = !check ? nothing :
        round.( defInt, readdlm( folderdata[i,j]*"frequency-map.txt" ) )
    next!( meter )
end

In [ ]:
# Plot the simulation success across the (ρ,s) pairs.
plt = plot(; size=(300,300))
plot!( plt; left_margin=0pt, bottom_margin=-5pt )

κdata = .!(Fdatalist .== nothing)
heatmap!( plt, ρlist, slist, κdata; color=colormap, colorbar=false )

plot!( plt; xlims=10.0.^[ρmin,ρmax], xscale=:log10  )
plot!( plt; ylims=10.0.^[smin,smax], yscale=:log10  )
plot!( plt; xlabel="density, "*L"ρ", ylabel="speed, "*L"s" )

In [ ]:
# Extract the clusters which can be optimized over for each parameter combination.
ιdatalist = [isnothing( Fdatalist[i,j] ) ? nothing : 0 .< vec( sum( Fdatalist[i,j]; dims=2 ) )
    for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ*Ns, "Solving for exponents." )

# Initialize data variables.
θdatalist = Matrix{Matrix{defFloat}}( undef, Nρ, Ns )

# Solve for the parameters.
θ0 = [1.5, 1.5, 0.66]
for i ∈ 1:1;  for j ∈ 1:1
    # Solve for the
    if κdata[i,j]
        θdatalist[i,j] = hcat( ThreadsX.map(
            k -> ιdatalist[i,j][k] ? convert( runopt( k, Fdatalist[i,j], N, θ0 ).minimizer ) : fill( -1.0, 3 ), 1:N )... )
    else
        θdatalist[i,j] = fill( -1.0, 3, N )
    end

    # Update progress meter.
    next!( meter )
end; end